In [ ]:
# --- Cell 1: environment + scratch space discovery -------------------------
import os, sys, platform, subprocess

print("project:", "FLUX.1-schnell")
print("python:", sys.version.split()[0])
print("platform:", platform.platform())

for p in ("/kaggle/temp", "/kaggle/working"):
    os.makedirs(p, exist_ok=True)
    st = os.statvfs(p)
    free = st.f_bavail * st.f_frsize / 1e9
    print(f"{p}: free={free:.1f} GB writable={os.access(p, os.W_OK)}")

print("--- GPU ---")
print(subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "no nvidia-smi")

print("--- internet ---")
for u in ("https://huggingface.co", "https://github.com"):
    r = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
                        "--max-time", "30", u], capture_output=True, text=True)
    print(u, "->", r.stdout)


In [ ]:
# --- Cell 2: install dependencies for FLUX GGUF inference ------------------
import subprocess, sys

pkgs = [
    "diffusers>=0.30.0",
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "sentencepiece",
    "gguf>=0.10.0",
    "protobuf"
]
print("installing:", " ".join(pkgs), flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("packages installed.", flush=True)


In [ ]:
# --- Cell 3: download FLUX.1-schnell GGUF and pipeline components ----------
import os, subprocess, sys
from huggingface_hub import hf_hub_download, snapshot_download

MODEL_REPO = "city96/FLUX.1-schnell-gguf"
MODEL_FILE = "flux1-schnell-Q4_K_S.gguf"
MODEL_SIZE_GB = float("6.32")
BASE_REPO = "Niansuh/FLUX.1-schnell"

MODEL_DIR = "/kaggle/temp/models"
BASE_DIR = "/kaggle/temp/base_pipe"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(BASE_DIR, exist_ok=True)

# 1. Download the quantized transformer GGUF from city96/FLUX.1-schnell-gguf
dest_model = os.path.join(MODEL_DIR, MODEL_FILE)
if not os.path.exists(dest_model) or os.path.getsize(dest_model) < 1000:
    print(f"Downloading {MODEL_FILE} from {MODEL_REPO} ...", flush=True)
    hf_hub_download(
        repo_id=MODEL_REPO,
        filename=MODEL_FILE,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False
    )
actual_size_gb = os.path.getsize(dest_model) / (1024**3)
print(f"Transformer weights: {dest_model} ({actual_size_gb:.2f} GB)", flush=True)

# 2. Download text encoders, tokenizer, VAE from ungated mirror (skip 24GB transformer)
print(f"Ensuring pipeline components from {BASE_REPO} ...", flush=True)
snapshot_download(
    repo_id=BASE_REPO,
    local_dir=BASE_DIR,
    ignore_patterns=["transformer/*.safetensors*", "*.sft", "*.jpeg", "*.jpg"],
    local_dir_use_symlinks=False
)
print("All model assets ready.", flush=True)


In [ ]:
# --- Cell 4: load FLUX pipeline + start server + open tunnels --------------
import os, re, subprocess, time, urllib.request, json, threading, http.server
import base64, io, gc
import torch
from diffusers import FluxPipeline, FluxTransformer2DModel, GGUFQuantizationConfig

PORT = int("8080")
CTL_PORT = int("8081")
DEFAULT_STEPS = int("4" or "4")
DEFAULT_WIDTH = int("1024" or "1024")
DEFAULT_HEIGHT = int("1024" or "1024")

STOP_REQUESTED = threading.Event()
GEN_LOCK = threading.Lock()

WORK = "/kaggle/temp/run"
os.makedirs(WORK, exist_ok=True)
CF_LOG = os.path.join(WORK, "cloudflared.log")
CTL_LOG = os.path.join(WORK, "cloudflared-ctl.log")

# ---- cloudflared binary ----
CF = "/kaggle/temp/cloudflared"
if not os.path.exists(CF):
    subprocess.run(["curl", "-sL", "-o", CF,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    os.chmod(CF, 0o755)

# ---- load pipeline ----
model_path = os.path.join("/kaggle/temp/models", "flux1-schnell-Q4_K_S.gguf")
base_dir = "/kaggle/temp/base_pipe"

# Ensure transformer/config.json exists locally so from_single_file never hits gated repo
trans_cfg_dir = os.path.join(base_dir, "transformer")
os.makedirs(trans_cfg_dir, exist_ok=True)
trans_cfg_file = os.path.join(trans_cfg_dir, "config.json")
if not os.path.exists(trans_cfg_file):
    print("Fetching transformer config.json from ungated mirror...", flush=True)
    try:
        url = "https://huggingface.co/Niansuh/FLUX.1-schnell/raw/main/transformer/config.json"
        with urllib.request.urlopen(url) as r:
            with open(trans_cfg_file, "wb") as f:
                f.write(r.read())
    except Exception as e:
        print("Warning: could not fetch config.json directly:", e, flush=True)

# Ensure scheduler_config.json exists (diffusers requires scheduler_config.json)
sched_dir = os.path.join(base_dir, "scheduler")
sched_cfg_file = os.path.join(sched_dir, "config.json")
sched_target = os.path.join(sched_dir, "scheduler_config.json")
if os.path.exists(sched_cfg_file) and not os.path.exists(sched_target):
    import shutil
    shutil.copyfile(sched_cfg_file, sched_target)

print("Loading GGUF transformer with float16 compute...", flush=True)
transformer = FluxTransformer2DModel.from_single_file(
    model_path,
    config=trans_cfg_dir,
    quantization_config=GGUFQuantizationConfig(compute_dtype=torch.float16),
    torch_dtype=torch.float16,
)

print("Loading FluxPipeline...", flush=True)
from diffusers import FlowMatchEulerDiscreteScheduler
scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(sched_dir)
pipe = FluxPipeline.from_pretrained(
    base_dir,
    transformer=transformer,
    scheduler=scheduler,
    torch_dtype=torch.float16,
)
pipe.enable_model_cpu_offload()
print("FLUX pipeline loaded and ready on GPU!", flush=True)

# ---- Control Server on CTL_PORT (port 8081) ----
STOP_TOKEN = os.environ.get("CLUSTER_STOP_TOKEN") or os.urandom(8).hex()

class CtlHandler(http.server.BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def _send(self, code, payload):
        body = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path.rstrip("/") in ("", "/health"):
            self._send(200, {"ok": True, "status": "up",
                             "stopping": STOP_REQUESTED.is_set(),
                             "model": "flux1-schnell-Q4_K_S.gguf"})
        elif self.path.startswith("/stop/" + STOP_TOKEN):
            STOP_REQUESTED.set()
            self._send(200, {"ok": True, "message": "shutdown requested"})
            print("STOP_REQUESTED received via control tunnel", flush=True)
        else:
            self._send(404, {"ok": False, "error": "not found"})

    def do_POST(self):
        self.do_GET()

ctl = http.server.ThreadingHTTPServer(("127.0.0.1", CTL_PORT), CtlHandler)
threading.Thread(target=ctl.serve_forever, daemon=True).start()

# ---- Image Generation Server on PORT (port 8080) ----
HTML_PAGE = """<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>FLUX.1-schnell Remote GPU Server</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; }
  body { background: #0f1117; color: #e1e4ea; min-height: 100vh; display: flex; flex-direction: column; align-items: center; padding: 24px 16px; }
  .card { background: #1a1d27; border: 1px solid #2d3139; border-radius: 12px; width: 100%; max-width: 840px; padding: 24px; box-shadow: 0 8px 24px rgba(0,0,0,0.4); margin-bottom: 24px; }
  h1 { font-size: 1.6rem; color: #fff; margin-bottom: 8px; display: flex; align-items: center; gap: 8px; }
  .badge { font-size: 0.75rem; background: #059669; color: #fff; padding: 3px 8px; border-radius: 6px; font-weight: 600; }
  p.subtitle { color: #9ca3af; font-size: 0.9rem; margin-bottom: 20px; }
  label { display: block; font-size: 0.85rem; font-weight: 600; color: #d1d5db; margin-bottom: 6px; }
  textarea { width: 100%; background: #0f1117; border: 1px solid #374151; border-radius: 8px; color: #fff; padding: 12px; font-size: 0.95rem; min-height: 90px; resize: vertical; margin-bottom: 16px; }
  textarea:focus { outline: none; border-color: #3b82f6; }
  .row { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 16px; margin-bottom: 16px; }
  select, input { width: 100%; background: #0f1117; border: 1px solid #374151; border-radius: 8px; color: #fff; padding: 10px; font-size: 0.9rem; }
  button { width: 100%; background: #2563eb; color: #fff; border: none; padding: 12px 20px; font-size: 1rem; font-weight: 600; border-radius: 8px; cursor: pointer; transition: background 0.2s; }
  button:hover { background: #1d4ed8; }
  button:disabled { background: #4b5563; cursor: not-allowed; }
  #status { margin-top: 16px; font-size: 0.9rem; color: #93c5fd; text-align: center; }
  #preview { margin-top: 24px; text-align: center; }
  #preview img { max-width: 100%; border-radius: 8px; box-shadow: 0 4px 16px rgba(0,0,0,0.5); }
  .download-btn { margin-top: 12px; display: inline-block; background: #10b981; color: #fff; text-decoration: none; padding: 8px 16px; border-radius: 6px; font-size: 0.9rem; font-weight: 600; }
  .api-info { background: #11131a; border: 1px solid #232733; border-radius: 8px; padding: 16px; margin-top: 24px; font-size: 0.85rem; }
  code { color: #60a5fa; font-family: ui-monospace, SFMono-Regular, monospace; }
  pre { background: #090a0f; padding: 10px; border-radius: 6px; overflow-x: auto; margin-top: 8px; color: #e5e7eb; }
</style>
</head>
<body>
<div class="card">
  <h1>FLUX.1-schnell <span class="badge">ONLINE</span></h1>
  <p class="subtitle">Fast 4-step diffusion transformer with GGUF Q4_K_S running on Kaggle GPU</p>
  
  <label for="prompt">Prompt</label>
  <textarea id="prompt" placeholder="e.g. A futuristic cyberpunk city at midnight, neon reflections on wet asphalt, volumetric mist, hyperrealistic, 8k"></textarea>
  
  <div class="row">
    <div>
      <label for="size">Resolution</label>
      <select id="size">
        <option value="1024x1024">1024 × 1024 (Square)</option>
        <option value="768x768">768 × 768</option>
        <option value="512x512">512 × 512 (Fastest)</option>
        <option value="1024x768">1024 × 768 (Landscape)</option>
        <option value="768x1024">768 × 1024 (Portrait)</option>
      </select>
    </div>
    <div>
      <label for="steps">Inference Steps (1-4)</label>
      <select id="steps">
        <option value="4" selected>4 (Recommended)</option>
        <option value="3">3</option>
        <option value="2">2</option>
        <option value="1">1 (Draft)</option>
      </select>
    </div>
    <div>
      <label for="seed">Seed (optional)</label>
      <input id="seed" type="number" placeholder="Random">
    </div>
  </div>
  
  <button id="genBtn" onclick="generate()">Generate Image</button>
  <div id="status"></div>
  <div id="preview"></div>

  <div class="api-info">
    <strong>OpenAI-compatible Image Generation Endpoint:</strong>
    <pre>POST /v1/images/generations
Content-Type: application/json

{"prompt": "A cybernetic cat on Mars", "size": "1024x1024", "response_format": "b64_json"}</pre>
  </div>
</div>

<script>
async function generate() {
  const prompt = document.getElementById('prompt').value.trim();
  if (!prompt) return alert('Please enter a prompt');
  const size = document.getElementById('size').value;
  const steps = parseInt(document.getElementById('steps').value);
  const seedVal = document.getElementById('seed').value;
  const btn = document.getElementById('genBtn');
  const status = document.getElementById('status');
  const preview = document.getElementById('preview');
  
  btn.disabled = true;
  preview.innerHTML = '';
  const startTime = Date.now();
  status.textContent = 'Generating on GPU... (~15-30s)';
  
  try {
    const payload = { prompt, size, steps, response_format: 'b64_json' };
    if (seedVal) payload.seed = parseInt(seedVal);
    const resp = await fetch('/v1/images/generations', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify(payload)
    });
    if (!resp.ok) {
      const err = await resp.text();
      throw new Error(`Server returned HTTP ${resp.status}: ${err}`);
    }
    const data = await resp.json();
    const b64 = data.data[0].b64_json;
    const elapsed = ((Date.now() - startTime) / 1000).toFixed(1);
    status.textContent = `Completed in ${elapsed}s!`;
    const imgSrc = 'data:image/png;base64,' + b64;
    preview.innerHTML = `
      <img src="${imgSrc}" alt="Generated Image"><br>
      <a class="download-btn" href="${imgSrc}" download="flux-${Date.now()}.png">Download Full Resolution</a>
    `;
  } catch (e) {
    status.textContent = 'Error: ' + e.message;
  } finally {
    btn.disabled = false;
  }
}
</script>
</body>
</html>
"""

class ServerHandler(http.server.BaseHTTPRequestHandler):
    def log_message(self, *a): pass

    def _send_json(self, code, payload):
        body = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(body)

    def do_OPTIONS(self):
        self.send_response(200)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type, Authorization")
        self.end_headers()

    def do_HEAD(self):
        self.do_GET()

    def do_GET(self):
        path = self.path.split("?")[0].rstrip("/")
        if path in ("", "/index.html"):
            body = HTML_PAGE.encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)
        elif path in ("/health", "/v1/health"):
            self._send_json(200, {
                "status": "ok",
                "model": "flux1-schnell",
                "architecture": "flux.1-schnell",
                "device": "cuda" if torch.cuda.is_available() else "cpu"
            })
        elif path in ("/models", "/v1/models"):
            self._send_json(200, {
                "data": [{
                    "id": "flux1-schnell",
                    "object": "model",
                    "type": "image-generation"
                }]
            })
        else:
            self._send_json(404, {"error": f"Path {path} not found"})

    def do_POST(self):
        path = self.path.split("?")[0].rstrip("/")
        content_len = int(self.headers.get("Content-Length", 0))
        post_data = self.rfile.read(content_len) if content_len > 0 else b"{}"
        try:
            req = json.loads(post_data.decode("utf-8"))
        except Exception:
            return self._send_json(400, {"error": "Invalid JSON payload"})

        if path in ("/v1/images/generations", "/generate"):
            prompt = req.get("prompt", "")
            if not prompt:
                return self._send_json(400, {"error": "Missing prompt"})

            size = req.get("size", "1024x1024")
            try:
                width, height = map(int, size.lower().split("x"))
            except Exception:
                width, height = DEFAULT_WIDTH, DEFAULT_HEIGHT

            steps = int(req.get("steps") or DEFAULT_STEPS)
            steps = max(1, min(4, steps))
            seed = req.get("seed")

            gen = None
            if seed is not None:
                try:
                    gen = torch.Generator("cpu").manual_seed(int(seed))
                except Exception:
                    pass

            t0 = time.time()
            print(f"[GEN] Prompt: '{prompt[:80]}...' size={width}x{height} steps={steps}", flush=True)

            with GEN_LOCK:
                with torch.inference_mode():
                    img = pipe(
                        prompt=prompt,
                        height=height,
                        width=width,
                        num_inference_steps=steps,
                        guidance_scale=0.0,
                        max_sequence_length=256,
                        generator=gen,
                    ).images[0]
                torch.cuda.empty_cache()
                gc.collect()

            elapsed = round(time.time() - t0, 2)
            print(f"[GEN] Done in {elapsed}s", flush=True)

            buf = io.BytesIO()
            img.save(buf, format="PNG")
            b64_bytes = base64.b64encode(buf.getvalue()).decode("utf-8")

            if path == "/generate":
                return self._send_json(200, {
                    "image_b64": b64_bytes,
                    "prompt": prompt,
                    "width": width,
                    "height": height,
                    "steps": steps,
                    "elapsed_seconds": elapsed
                })
            else:
                return self._send_json(200, {
                    "created": int(time.time()),
                    "data": [
                        {
                            "b64_json": b64_bytes,
                            "revised_prompt": prompt
                        }
                    ]
                })
        else:
            self._send_json(404, {"error": f"POST {path} not supported"})

server = http.server.ThreadingHTTPServer(("127.0.0.1", PORT), ServerHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("Image server running on port", PORT, flush=True)

# ---- publish both tunnels ----
def start_tunnel(url, log_path):
    fh = open(log_path, "w")
    subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", url],
                     stdout=fh, stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            return m.group(0)
    return None

public_url = start_tunnel(f"http://127.0.0.1:{PORT}", CF_LOG)
control_url = start_tunnel(f"http://127.0.0.1:{CTL_PORT}", CTL_LOG)

if not public_url:
    print("cloudflared log:", open(CF_LOG).read()[-2000:], flush=True)
    raise RuntimeError("could not obtain a public tunnel URL")
if not control_url:
    print("control cloudflared log:", open(CTL_LOG).read()[-2000:], flush=True)
    raise RuntimeError("could not obtain a control tunnel URL")

with open("/kaggle/working/CONTROL_URL.txt", "w") as f:
    f.write(public_url + "\n")
with open("/kaggle/working/STOP_URL.txt", "w") as f:
    f.write(control_url + "/stop/" + STOP_TOKEN + "\n")

print("=" * 72)
print("CONTROL_URL=" + public_url)
print("STOP_URL=" + control_url + "/stop/" + STOP_TOKEN)
print("MODEL=" + "flux1-schnell-Q4_K_S.gguf")
print("=" * 72)
print(f"Web UI         : {public_url}")
print(f"API endpoint   : {public_url}/v1/images/generations")
print(f"Health         : {public_url}/health")
print("Stop from PC   : ctr.sh stop   (uses the control tunnel above)")


In [ ]:
# --- Cell 5: keep the tunnel alive (this is what you keep running) ---------
import time, urllib.request

PORT = int("8080")
SERVE_MINUTES = int("660")

url = open("/kaggle/working/CONTROL_URL.txt").read().strip()
print(f"serving for ~{SERVE_MINUTES} minutes at {url}", flush=True)

for minute in range(SERVE_MINUTES):
    # Wait up to 60s, but poll the stop flag every 5s so shutdown is prompt.
    for _ in range(12):
        if STOP_REQUESTED.is_set():
            break
        time.sleep(5)
    if STOP_REQUESTED.is_set():
        print("STOP received from control tunnel -> ending session", flush=True)
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as r:
            status = str(r.status)
    except Exception as e:
        status = f"err {e}"
    if minute % 5 == 0:
        # Re-print periodically so the URL can always be recovered from logs.
        print(f"[{minute}m] health={status} CONTROL_URL={url}", flush=True)

print("SERVE_WINDOW_COMPLETE", flush=True)
